In [ ]:
library(ArchR)
library(BSgenome.Hsapiens.UCSC.hg38)
library(Seurat)
library(dplyr)
set.seed(1)

addArchRThreads(threads = 42) 
addArchRGenome("hg38")

setwd('/Aging/Data/Anderson/Multiomics/ATAC/ArchR/')

#----------------------------------------------------
## Creating ArrowFiles
set.seed(1)
addArchRThreads(threads = 16) 

inputFiles <- c('GSE214979_atac_fragments' = '/Aging/Data/Anderson/Multiomics/Raw/GSE214979_atac_fragments.tsv.gz')
inputFiles
addArchRGenome("hg38")

setwd('/Aging/Data/Anderson/Multiomics/ATAC/ArchR/')

ArrowFiles <- createArrowFiles(
  inputFiles = inputFiles,
  sampleNames = names(inputFiles),
  filterTSS = 4, #Dont set this too high because you can always increase later
  filterFrags = 1000, 
  addTileMat = TRUE,
  addGeneScoreMat = TRUE
)
ArrowFiles

#----------------------------------------------------
## Inferring Doublets 
ArrowFiles = "GSE214979_atac_fragments.arrow"

doubScores <- addDoubletScores(
  input = ArrowFiles,
  k = 10, #Refers to how many cells near a "pseudo-doublet" to count.
  knnMethod = "UMAP", #Refers to the embedding to use for nearest neighbor search.
  LSIMethod = 1
)

## Creating an ArchRProject
proj <- ArchRProject(
  ArrowFiles = ArrowFiles, 
  outputDirectory = "ArchRProject",
  copyArrows = TRUE #This is recommened so that you maintain an unaltered copy for later usage.
)
proj <- filterDoublets(ArchRProj = proj)
getAvailableMatrices(proj)

#----------------------------------------------------

#saveArchRProject(ArchRProj = proj, outputDirectory = "ArchRProject", load = FALSE)
proj <- loadArchRProject(path = "/Aging/Data/Anderson/Multiomics/ATAC/ArchR/ArchRProject", force = FALSE, showLogo = FALSE)

getAvailableMatrices(proj)
head(proj$cellNames)
head(proj$Sample)
quantile(proj$TSSEnrichment)

#----------------------------------------------------

## Adding coldata 
df = as.data.frame(proj$cellNames)
colnames(df) <- 'X'
# sort new column to the order of cellNames in ArchR project
metadata = read.csv('/Aging/Data/Anderson/Multiomics/Raw/GSE214979_cell_metadata.csv')
# AUCell obj 
data_dir2 <- '/Aging/Data/Anderson/snRNA_Seq/'
query <- readRDS(paste0(data_dir2,"GSE214979_filtered_feature_bc_matrix_processed_seurat_neurons_only_with_Ruzicka_AUCell_M0_M9_M24_dd.rds"))
metadata <- left_join(metadata,query@meta.data[c(5,27,60:81)], by = 'X')

metadata$geneset0_status <- coalesce(metadata$geneset_pos0.y, metadata$geneset_neg0.y)
metadata$geneset24_status <- coalesce(metadata$geneset_pos24.y, metadata$geneset_neg24.y)
#metadata$geneset0_cr_status <- coalesce(metadata$geneset_pos0_cr.y, metadata$geneset_neg0_cr.y)
#metadata$geneset9_cr_status <- coalesce(metadata$geneset_pos9_cr.y, metadata$geneset_neg9_cr.y)

q1 = round(quantile(metadata$geneset_dd.x,0.25,na.rm=T),4)
q2 = round(quantile(metadata$geneset_dd.x,0.75,na.rm=T),4)
metadata = metadata %>% mutate(geneset_dd_status = case_when(geneset_dd.x <= q1 ~ "low",
                                                 geneset_dd.x >= q2 ~ "high"))

metadata$X = paste0('GSE214979_atac_fragments#',metadata$X)
metadata1 = left_join(df, metadata, by = 'X')

names = colnames(metadata1)[-1]
# loop through through columns
for (i in 1:length(names)) {
    print(names[i])
    proj <- addCellColData(ArchRProj = proj, data = paste0(metadata1[[names[i]]]),
                           cells = metadata1$X, name = names[i], force = TRUE)}
#getCellColData(proj)